# Phase 0 — VRAM spike

**One question:** does LatentSync 1.6 inference at 512 fit in Kaggle's 16 GB?

The whole 1080p plan rests on this. LatentSync 1.6 was retrained on 512×512
video to fix 1.5's blurry teeth and lips, and that larger face crop is what
makes a 1080p render worthwhile. If it does not fit, the fallback is 1.5 at 256
with a 720p output — which is a perfectly good plan, just a different one.

The widely-quoted "18 GB" and "30 GB" figures for LatentSync are **training**
requirements. Inference is far lighter. But "far lighter" is not a number, so
this notebook measures it.

## Before you run

1. **Settings → Accelerator → GPU** (P100 or T4 ×2). Without this the notebook
   exits immediately.
2. **Settings → Internet → On** (needed to clone the repo and pull weights).
3. Attach your private `talkinghead-assets` dataset via **+ Add Input** if you
   have one. Optional — the notebook falls back to a public sample clip.

Runtime is roughly 15–25 minutes, most of it downloading weights. It draws
against your 30 hr/week GPU quota, so this costs about 0.4 hours of it.

**Read the verdict in the last cell.** It tells you which profile to use.

## 1. Confirm the GPU

Fail fast: a CPU session would spend twenty minutes downloading weights before
discovering it cannot answer the question.

In [ ]:
import subprocess, sys

try:
    import torch
except ImportError:
    sys.exit("torch is not available in this session.")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU in this session.\n"
        "Settings -> Accelerator -> GPU (P100 or T4 x2), then re-run."
    )

props = torch.cuda.get_device_properties(0)
TOTAL_VRAM_GB = props.total_memory / 1024**3

print(f"GPU            {props.name}")
print(f"total VRAM     {TOTAL_VRAM_GB:.2f} GB")
print(f"torch          {torch.__version__}")
print(f"CUDA           {torch.version.cuda}")
print(f"visible GPUs   {torch.cuda.device_count()}")
print()
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
    capture_output=True, text=True,
).stdout)

## 2. Clone LatentSync from the first-party repo

`github.com/bytedance/LatentSync` only. Apache 2.0 on code *and* weights, which
is why this project can use it for client work at all. No mirrors, no ComfyUI
wrappers, no re-uploads.

In [ ]:
import os
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "LatentSync"

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/bytedance/LatentSync.git {REPO_DIR}

os.chdir(REPO_DIR)
!git log -1 --format="pinned commit: %H%n            %s"

# Record the SHA -- once this spike passes, freeze it in config.TRUSTED_SOURCES
# so a future upstream change cannot silently alter behaviour.
LATENTSYNC_SHA = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=REPO_DIR
).stdout.strip()
print(f"\nSHA to pin: {LATENTSYNC_SHA}")

## 3. Select the inference config

The repo ships **two** configs declaring 512: `stage1_512.yaml` and
`stage2_512.yaml`. Only the second is the inference config — ByteDance's own
`inference.sh` uses `configs/unet/stage2_512.yaml`, and stage1 is an earlier
*training* stage. Picking purely by declared resolution chooses wrongly, because
both say 512.

So: prefer `stage2`, prefer the `_512` variant, and verify the resolution rather
than assuming it. The shipped config is used as-is — no rewriting, since
`stage2_512.yaml` already declares 512 and its relative paths
(`syncnet_config_path`, `mask_image_path`) resolve against the repo root.

In [ ]:
import yaml

TARGET_RESOLUTION = 512

# Matches ByteDance's own inference.sh. Resolved as a path rather than hardcoded
# blindly, so a rename surfaces here with the full listing instead of as a
# confusing failure inside the subprocess.
PREFERRED = f"stage2_{TARGET_RESOLUTION}.yaml"

unet_dir = REPO_DIR / "configs" / "unet"
available = sorted(unet_dir.glob("*.yaml"))
print("U-Net configs shipped:")
for c in available:
    print(f"  {c.name}")


def declared_resolution(path):
    """Pull the resolution out of a config, wherever it is nested."""
    try:
        data = yaml.safe_load(path.read_text())
    except Exception:
        return None
    stack = [data]
    while stack:
        node = stack.pop()
        if isinstance(node, dict):
            for key, value in node.items():
                if key == "resolution" and isinstance(value, int):
                    return value
                stack.append(value)
        elif isinstance(node, list):
            stack.extend(node)
    return None


UNET_CONFIG = unet_dir / PREFERRED
if not UNET_CONFIG.is_file():
    # Fall back to any stage2 config, never stage1 -- stage1 is a training stage
    # and would silently measure the wrong thing.
    stage2 = [c for c in available if c.name.startswith("stage2")]
    if not stage2:
        raise SystemExit(
            f"No stage2 config found in {unet_dir}. Available: "
            f"{[c.name for c in available]}"
        )
    UNET_CONFIG = max(stage2, key=lambda c: declared_resolution(c) or 0)
    print(f"\n{PREFERRED} not present; falling back to {UNET_CONFIG.name}")

SPIKE_CONFIG = UNET_CONFIG
RESOLUTION = declared_resolution(SPIKE_CONFIG)

print(f"\nusing:      {SPIKE_CONFIG.relative_to(REPO_DIR)}")
print(f"resolution: {RESOLUTION}")

if RESOLUTION != TARGET_RESOLUTION:
    print(
        f"\nWARNING: expected {TARGET_RESOLUTION}, config declares {RESOLUTION}.\n"
        f"The measurement below will describe {RESOLUTION}, not "
        f"{TARGET_RESOLUTION}. Interpret the verdict accordingly."
    )

In [ ]:
# Show the settings that actually drive VRAM use. batch_size and num_frames
# matter as much as resolution: peak memory is set by how many 512x512 frames
# are in flight at once, which is why a 5-second clip measures the same peak a
# three-minute one would.
cfg = yaml.safe_load(SPIKE_CONFIG.read_text())
data = cfg.get("data", {})
run = cfg.get("run", {})

print(f"config       {SPIKE_CONFIG.relative_to(REPO_DIR)}")
print(f"resolution   {data.get('resolution')}")
print(f"batch_size   {data.get('batch_size')}")
print(f"num_frames   {data.get('num_frames')}")
print(f"video_fps    {data.get('video_fps')}")
print()
print(f"inference_steps  {run.get('inference_steps')}")
print(f"guidance_scale   {run.get('guidance_scale')}")

## 4. Pull the 1.6 weights

From `hf.co/ByteDance/LatentSync-1.6`. This is the slow part — several GB.

Once the spike passes, copy these files into your private Kaggle Dataset. Every
later session then mounts them read-only instead of re-downloading, which is the
single biggest practical speedup in the whole project.

In [ ]:
!pip install --quiet --upgrade huggingface_hub

from huggingface_hub import list_repo_files, snapshot_download

HF_REPO = "ByteDance/LatentSync-1.6"

print(f"files in {HF_REPO}:")
for f in sorted(list_repo_files(HF_REPO)):
    print(f"  {f}")

In [ ]:
CKPT_DIR = REPO_DIR / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# The repo's inference script expects weights under checkpoints/, including the
# Whisper audio encoder and the auxiliary face-detection models.
snapshot_download(repo_id=HF_REPO, local_dir=str(CKPT_DIR), local_dir_use_symlinks=False)

print("\ndownloaded:")
total = 0
for p in sorted(CKPT_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1024**2
        total += size_mb
        if size_mb > 1:
            print(f"  {size_mb:9.1f} MB  {p.relative_to(CKPT_DIR)}")
print(f"  {total:9.1f} MB  TOTAL")

UNET_CKPT = next(CKPT_DIR.rglob("latentsync_unet.pt"), None)
if UNET_CKPT is None:
    candidates = [p for p in CKPT_DIR.rglob("*.pt")] + [p for p in CKPT_DIR.rglob("*.ckpt")]
    raise SystemExit(f"latentsync_unet.pt not found. Candidates: {candidates}")
print(f"\nU-Net checkpoint: {UNET_CKPT.relative_to(REPO_DIR)}")

## 5. Install dependencies

Drawn from the repo's own `requirements.txt`, with deliberate departures:

- **torch / torchvision excluded.** Kaggle ships 2.10.0+cu128; the repo pins
  2.5.1+cu121. Installing that pin would tear down a working CUDA stack.
- **numpy excluded.** The repo pins 1.26.4, and downgrading numpy on Kaggle
  breaks a long list of preinstalled packages. Left alone unless something
  actually complains.
- **gradio excluded.** A web UI we never launch.
- **Most pins dropped.** `mediapipe==0.10.11` has no release for this Python, and
  because pip aborts the whole command on one unresolvable package, that single
  pin silently prevented all twenty from installing on the previous run. Only
  `diffusers` and `transformers` stay pinned, since LatentSync depends on APIs
  that moved.

Installed **one package at a time** for the same reason: a single failure should
cost you that one package, not the entire environment. Each outcome is printed.


In [ ]:
# One unavailable pin aborts the ENTIRE pip command, taking every other package
# with it. That is what happened on the previous run: mediapipe==0.10.11 has no
# release for this Python (only 0.10.13+), so all twenty packages were skipped
# and the failure looked like a single harmless error line.
#
# So: install one package at a time, report each outcome, and pin only where the
# pin actually matters. diffusers and transformers stay pinned because LatentSync
# uses APIs that moved; the rest float to whatever resolves here.
PACKAGES = [
    "diffusers==0.32.2",       # pinned: LatentSync uses 0.32-era APIs
    "transformers==4.48.0",    # pinned: paired with the diffusers version
    "decord==0.6.0",
    "accelerate",
    "einops",
    "omegaconf",
    "opencv-python",
    "mediapipe",               # unpinned: 0.10.11 does not exist for this Python
    "python_speech_features",
    "librosa",
    "scenedetect",
    "ffmpeg-python",
    "imageio",
    "imageio-ffmpeg",
    "lpips",
    "face-alignment",
    "kornia",
    "insightface==0.7.3",      # pinned: 0.7.3 is what upstream tests against
    "onnxruntime-gpu",
    "DeepCache==0.1.1",        # required by scripts/inference.py for --enable_deepcache
    "soundfile",
]

# insightface builds a Cython extension, so its build environment needs cython
# and numpy headers available first.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "cython"],
               capture_output=True, text=True)

failed = []
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"  ok      {pkg}")
    else:
        failed.append(pkg)
        tail = result.stderr.strip().splitlines()
        reason = tail[-1][:100] if tail else "unknown"
        print(f"  FAILED  {pkg}  --  {reason}")

print()
if failed:
    print(f"{len(failed)} package(s) failed: {', '.join(failed)}")
    print("The probe below will report whether that actually breaks anything.")
else:
    print("All packages installed.")

# ffmpeg is host-provided on Kaggle -- confirm rather than install.
print()
subprocess.run(["bash", "-c", "ffmpeg -version | head -1"])


### 5b. Probe the real entry point

The previous probe imported `LipsyncPipeline` and reported green — while
`DeepCache` was still missing. It was checking the wrong thing: `DeepCache` is
imported by `scripts/inference.py`, not by the pipeline module.

So this now runs `python -m scripts.inference --help`, which exercises *every*
module-level import of the actual entry point without running any inference. If
that exits cleanly, the real command cannot fail on a missing module.

It loops, installing whatever each traceback names, so a chain of missing
packages resolves in one pass rather than one per notebook run.


In [ ]:
import re

# pip name differs from import name for several of these, so a missing module
# has to be translated back into something installable.
IMPORT_TO_PIP = {
    "cv2": "opencv-python",
    "face_alignment": "face-alignment",
    "python_speech_features": "python_speech_features",
    "onnxruntime": "onnxruntime-gpu",
    "DeepCache": "DeepCache==0.1.1",
    "ffmpeg": "ffmpeg-python",
    "skimage": "scikit-image",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "yaml": "pyyaml",
    "insightface": "insightface==0.7.3",
    "decord": "decord==0.6.0",
}

# --help exits after argparse, so this exercises all module-level imports of the
# real entry point without doing any work on the GPU.
PROBE_CMD = [sys.executable, "-m", "scripts.inference", "--help"]

installed = []
for attempt in range(12):
    proc = subprocess.run(PROBE_CMD, cwd=REPO_DIR, capture_output=True, text=True)
    if proc.returncode == 0:
        break

    match = re.search(r"No module named '([^']+)'", proc.stderr)
    if not match:
        print("Failed for a reason other than a missing module:\n")
        print("\n".join(proc.stderr.strip().splitlines()[-25:]))
        break

    module = match.group(1).split(".")[0]
    if module in installed:
        print(f"'{module}' still missing after installing it -- stopping.")
        break
    installed.append(module)

    pkg = IMPORT_TO_PIP.get(module, module)
    print(f"missing '{module}' -> installing {pkg}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        tail = result.stderr.strip().splitlines()
        print(f"  install FAILED: {tail[-1][:120] if tail else 'unknown'}")
else:
    print("Gave up after 12 attempts.")

IMPORTS_OK = proc.returncode == 0
print()
if IMPORTS_OK:
    print("Entry point imports cleanly -- scripts.inference --help succeeded.")
    if installed:
        print(f"(had to install: {', '.join(installed)})")
else:
    print("BROKEN. The measurement below will not run.")
    print("\n".join(proc.stderr.strip().splitlines()[-25:]))


## 6. Build a 5-second test clip

Uses your real base loop if the dataset is attached, since that is the footage
the answer actually needs to apply to. Falls back to the repo's own sample.

Five seconds is enough: LatentSync processes video in fixed-size batches, so peak
VRAM is set by batch size and resolution, **not** by total video length. A short
clip measures the same peak a three-minute one would.

In [ ]:
SPIKE_DIR = WORK / "spike"
SPIKE_DIR.mkdir(exist_ok=True)
TEST_VIDEO = SPIKE_DIR / "test_5s.mp4"
TEST_AUDIO = SPIKE_DIR / "test_5s.wav"

asset_video = None
for candidate in Path("/kaggle/input").rglob("base_loop.mp4"):
    asset_video = candidate
    break

if asset_video:
    print(f"using your footage: {asset_video}")
    !ffmpeg -y -loglevel error -i "{asset_video}" -t 5 -an -c:v libx264 -crf 16 -pix_fmt yuv420p "{TEST_VIDEO}"
else:
    sample = next((REPO_DIR / "assets").rglob("*.mp4"), None) if (REPO_DIR / "assets").exists() else None
    if sample:
        print(f"dataset not attached; using repo sample: {sample.name}")
        !ffmpeg -y -loglevel error -i "{sample}" -t 5 -an -c:v libx264 -crf 16 -pix_fmt yuv420p "{TEST_VIDEO}"
    else:
        raise SystemExit(
            "No test footage. Attach the talkinghead-assets dataset via + Add Input.\n"
            "A synthetic clip will not work -- LatentSync needs a real detectable face."
        )

# Any speech works for a VRAM measurement; a tone is enough to drive the model.
asset_audio = next(Path("/kaggle/input").rglob("reference.wav"), None)
if asset_audio:
    !ffmpeg -y -loglevel error -i "{asset_audio}" -t 5 -ac 1 -ar 16000 "{TEST_AUDIO}"
else:
    !ffmpeg -y -loglevel error -f lavfi -i "sine=frequency=200:sample_rate=16000" -t 5 -ac 1 "{TEST_AUDIO}"

!ffprobe -v error -show_entries stream=width,height,duration,codec_type -of default=noprint_wrappers=1 "{TEST_VIDEO}"
!ffprobe -v error -show_entries stream=duration,sample_rate -of default=noprint_wrappers=1 "{TEST_AUDIO}"

## 7. Measure — sweep to find what fits

The first real run answered the original question: **512 with `num_frames: 16`
peaks at 13.65 GB and OOMs on a 14.56 GiB T4**, roughly 1 GB short.

Rather than stop at "does not fit", this sweeps the one setting that actually
controls the peak. The traceback puts the OOM inside `vae.encode(masked_image)`,
which encodes all `num_frames` images as a single batch — so halving `num_frames`
roughly halves that allocation. It is a config value, not a code change.

The sweep walks 16 → 8 → 4 and stops at the first that completes, recording peak
VRAM for each. Higher `num_frames` means more temporal context and less risk of
flicker between windows, so the largest value that fits is the one to keep.

`PYTORCH_ALLOC_CONF=expandable_segments:True` is set as well — the OOM message
recommends it, and it costs nothing. It only reclaims fragmentation, and just
161 MB was reserved-but-unallocated here, so do not expect it to matter much on
its own.

**If everything here fails, that is a clean answer too:** use `TH_PROFILE=720p`
with the LatentSync 1.5 checkpoint at 256.


In [ ]:
import threading, time

class VramPoller:
    """Samples total GPU memory in use and remembers the peak."""

    def __init__(self, interval=0.25):
        self.interval = interval
        self.peak_mb = 0.0
        self.samples = []
        self._stop = threading.Event()
        self._thread = None

    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5,
                ).stdout.strip().splitlines()
                used = max(float(v) for v in out if v.strip())
                self.samples.append(used)
                self.peak_mb = max(self.peak_mb, used)
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        if self._thread:
            self._thread.join(timeout=5)

    @property
    def peak_gb(self):
        return self.peak_mb / 1024


# Baseline before loading anything, so the model's own cost is separable from
# whatever the session already had resident.
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
baseline = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
).stdout.strip().splitlines()
BASELINE_GB = max(float(v) for v in baseline if v.strip()) / 1024
print(f"baseline VRAM in use: {BASELINE_GB:.2f} GB")

In [ ]:
import copy
import os

INFERENCE_STEPS = 20
GUIDANCE_SCALE = 1.5

# Descending: the first that fits is the best one, since more temporal context
# means less flicker between windows.
NUM_FRAMES_SWEEP = [16, 8, 4]

if not IMPORTS_OK:
    raise SystemExit(
        "Entry point imports are broken -- fix cell 5b first. Running anyway "
        "only reproduces the same ModuleNotFoundError and reports 0.00 GB."
    )

# Recommended by the OOM message itself. Only reclaims fragmentation, so it is
# unlikely to be decisive here -- but it is free.
env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True")

base_cfg = yaml.safe_load(SPIKE_CONFIG.read_text())
results = []

for num_frames in NUM_FRAMES_SWEEP:
    # Write a variant config. Relative paths inside it (syncnet_config_path,
    # mask_image_path) resolve against cwd, which is REPO_DIR below.
    variant = copy.deepcopy(base_cfg)
    variant["data"]["num_frames"] = num_frames
    variant_path = WORK / f"unet_512_nf{num_frames}.yaml"
    variant_path.write_text(yaml.safe_dump(variant, sort_keys=False))

    out_video = SPIKE_DIR / f"out_512_nf{num_frames}.mp4"
    cmd = [
        sys.executable, "-m", "scripts.inference",
        "--unet_config_path", str(variant_path),
        "--inference_ckpt_path", str(UNET_CKPT),
        "--inference_steps", str(INFERENCE_STEPS),
        "--guidance_scale", str(GUIDANCE_SCALE),
        "--enable_deepcache",
        "--video_path", str(TEST_VIDEO),
        "--audio_path", str(TEST_AUDIO),
        "--video_out_path", str(out_video),
    ]

    print("=" * 68)
    print(f"num_frames = {num_frames}")
    print("=" * 68)

    started = time.time()
    with VramPoller() as poller:
        proc = subprocess.run(
            cmd, cwd=REPO_DIR, capture_output=True, text=True, env=env
        )
    elapsed = time.time() - started

    combined = (proc.stdout + proc.stderr).lower()
    oom = "out of memory" in combined
    ok = proc.returncode == 0 and out_video.exists()

    results.append({
        "num_frames": num_frames,
        "peak_gb": poller.peak_gb,
        "elapsed": elapsed,
        "oom": oom,
        "success": ok,
        "out": out_video,
        "stderr": proc.stderr,
    })

    print(f"  exit code   {proc.returncode}")
    print(f"  elapsed     {elapsed:.0f}s for 5s of video")
    print(f"  peak VRAM   {poller.peak_gb:.2f} GB")
    print(f"  OOM         {oom}")
    print(f"  output      {out_video.exists()}")

    if ok:
        print(f"\n  SUCCEEDED at num_frames={num_frames} -- stopping the sweep.")
        break
    if not oom:
        # A non-memory failure will not be fixed by shrinking the batch, so
        # stop rather than burning GPU quota on three identical errors.
        print("\n  Failed for a reason other than memory -- stopping the sweep.")
        print("\n".join(proc.stderr.strip().splitlines()[-25:]))
        break
    print("  OOM -- trying a smaller num_frames.\n")

# Carry the best result forward for the verdict cell.
winner = next((r for r in results if r["success"]), None)
SUCCESS = winner is not None
if winner:
    PEAK_GB = winner["peak_gb"]
    elapsed = winner["elapsed"]
    NUM_FRAMES = winner["num_frames"]
    OUT_VIDEO = winner["out"]
    OOM = False
else:
    worst = results[-1]
    PEAK_GB = max(r["peak_gb"] for r in results)
    elapsed = worst["elapsed"]
    NUM_FRAMES = worst["num_frames"]
    OUT_VIDEO = worst["out"]
    OOM = any(r["oom"] for r in results)

MEASURED = PEAK_GB >= 0.5
print()
print("sweep summary:")
for r in results:
    status = "OK" if r["success"] else ("OOM" if r["oom"] else "ERROR")
    print(f"  num_frames={r['num_frames']:3d}  peak {r['peak_gb']:5.2f} GB  "
          f"{r['elapsed']:5.0f}s  {status}")
if not MEASURED:
    print("\nNOTE: peak near zero -- nothing was measured. Not headroom.")


## 8. Verdict

A margin is reserved rather than accepting "it fit once." Real renders run longer
than five seconds and Kaggle sessions have other things resident, so a result
that only just fits will fail intermittently — which is worse than choosing the
720p profile deliberately.

In [ ]:
SAFETY_MARGIN_GB = 1.5
usable = TOTAL_VRAM_GB - SAFETY_MARGIN_GB

print("=" * 68)
print("PHASE 0 VERDICT")
print("=" * 68)
print(f"  GPU              {props.name}")
print(f"  total VRAM       {TOTAL_VRAM_GB:.2f} GB")
print(f"  usable budget    {usable:.2f} GB  (after {SAFETY_MARGIN_GB} GB margin)")
print()

if SUCCESS:
    print(f"  512 RUNS at num_frames={NUM_FRAMES}")
    print(f"  peak observed    {PEAK_GB:.2f} GB")
    print(f"  render speed     {elapsed / 5:.1f}s compute per 1s of video")
    print(f"  2-minute render  ~{elapsed / 5 * 120 / 60:.0f} minutes of GPU time")
    print()
    if PEAK_GB <= usable:
        print(f"  RESULT: fits with {usable - PEAK_GB:.2f} GB of margin.")
        print()
        print("  -> Use the 1080p profile. Set num_frames in the LatentSync")
        print(f"     config to {NUM_FRAMES} when the provider is built.")
        print(f"  -> Freeze this commit in config.TRUSTED_SOURCES:")
        print(f"       latentsync revision = {LATENTSYNC_SHA}")
        VERDICT = "1080p"
    else:
        print(f"  RESULT: ran, but peak {PEAK_GB:.2f} GB exceeds the "
              f"{usable:.2f} GB budget.")
        print()
        print("  -> It completed, but with under 1.5 GB spare a longer render")
        print("     on a busier session will OOM intermittently. That is worse")
        print("     to debug than choosing 720p deliberately.")
        print("  -> Either drop num_frames one more step, or set TH_PROFILE=720p.")
        VERDICT = "1080p-tight"
else:
    print(f"  highest peak     {PEAK_GB:.2f} GB")
    print()
    if OOM:
        print("  RESULT: 512 does not fit at any num_frames tried.")
        print()
        print("  -> Set TH_PROFILE=720p and use the LatentSync 1.5 checkpoint")
        print("     at 256. This is the planned fallback, not a failure:")
        print("     downscaling 1080p source to 720p pulls the whole frame")
        print("     toward the mouth's real resolution, which hides most of")
        print("     the softness from the smaller crop.")
        print()
        print("  -> Worth one more try first if you want 1080p: switch the")
        print("     accelerator to P100 (16280 MiB vs the T4's 15360 MiB,")
        print("     about 0.9 GB more) and re-run.")
        VERDICT = "720p"
    else:
        print("  RESULT: failed for a reason other than memory.")
        print("  -> Read the stderr above. A missing dependency or an")
        print("     undetected face looks nothing like an OOM, and no profile")
        print("     change will fix it.")
        VERDICT = "inconclusive"

print()
print(f"  VERDICT: {VERDICT}")
print("=" * 68)


## 9. Look at the output

The number answers whether it *runs*. Only your eyes answer whether it is good.

Watch the mouth specifically: do the teeth read as teeth at full resolution, or
as a smear? That is the exact failure 1.6 was trained to fix, so this is where
you confirm it actually did.

In [ ]:
from IPython.display import Video, display

if OUT_VIDEO.exists():
    !ffprobe -v error -show_entries stream=width,height,nb_frames -of default=noprint_wrappers=1 "{OUT_VIDEO}"
    # Crop to the face region and blow it up, so mouth detail is actually
    # visible rather than lost in a small inline player.
    ZOOM = SPIKE_DIR / "out_512_mouth.mp4"
    !ffmpeg -y -loglevel error -i "{OUT_VIDEO}" -vf "crop=iw/3:ih/3:iw/3:ih/2,scale=640:-2" -an "{ZOOM}"
    print("\nfull frame:")
    display(Video(str(OUT_VIDEO), embed=True, width=720))
    print("mouth region, zoomed:")
    display(Video(str(ZOOM), embed=True, width=640))
else:
    print("No output video -- see the failure output above.")

## What to do next

**If the verdict was `1080p`:** nothing to change — that is the default profile.
Freeze the printed commit SHA into `config.TRUSTED_SOURCES`, copy
`checkpoints/` into your private Kaggle Dataset so future sessions skip the
download, and move on to Phase 2 (the Chatterbox voice provider).

**If the verdict was `720p`:** set `TH_PROFILE=720p` and carry on. This was the
planned fallback, not a failure — you still get a good video, and the 1080p
source downscaling to 720p is what conceals the smaller face crop.

**If it was `inconclusive`:** the failure was not about memory. Read the stderr
in cell 7 before changing anything.

Either way, save this notebook's output — it is the evidence behind the profile
choice, and worth having when someone asks why.